In [1]:
from strands.tools import tool
from strands_tools import retrieve

In [2]:
@tool
def search_products(category: str, budget: str = None, brand: str = None) -> str:
    """
    Search for products based on category, budget, and brand preferences.
    """

    print("category =", category)
    print("budget =", budget)
    print("brand =", brand)

    # Mock product catalog
    products = [
        {"name": "Dell XPS 13", "category": "laptop", "brand": "Dell", "price": 95000},
        {"name": "MacBook Air M2", "category": "laptop", "brand": "Apple", "price": 110000},
        {"name": "HP Pavilion 15", "category": "laptop", "brand": "HP", "price": 70000},
        {"name": "Lenovo Legion 5", "category": "laptop", "brand": "Lenovo", "price": 98000},
    ]

    # Normalize input
    category = category.lower()

    # Flexible category matching
    filtered = [
        p for p in products
        if p["category"].lower() in category
        or category in p["category"].lower()
    ]

    # Brand filtering
    if brand:
        filtered = [
            p for p in filtered
            if p["brand"].lower() == brand.lower()
        ]

    if not filtered:
        return "No products found matching your criteria."

    result = "Recommended Products:\n\n"

    for p in filtered:
        result += (
            f"• {p['name']} ({p['brand']}) - ₹{p['price']}\n"
        )

    return result


print("✅ Product search tool ready")

✅ Product search tool ready


In [3]:
@tool
def get_product_details(product_name: str) -> str:
    """
    Get detailed specifications of a product.

    Args:
        product_name: Name of the product

    Returns:
        Product specifications and features
    """

    mock_db = {
        "Dell XPS 13": {
            "processor": "Intel i7",
            "ram": "16GB",
            "storage": "512GB SSD",
            "weight": "1.2kg",
            "battery": "12 hours",
            "display": "13.4-inch FHD+",
            "os": "Windows 11",
            "price": "₹95,000"
        },

        "MacBook Air M2": {
            "processor": "Apple M2",
            "ram": "16GB",
            "storage": "512GB SSD",
            "weight": "1.24kg",
            "battery": "18 hours",
            "display": "13.6-inch Liquid Retina",
            "os": "macOS",
            "price": "₹1,10,000"
        },

        "HP Pavilion 15": {
            "processor": "Intel i5",
            "ram": "16GB",
            "storage": "512GB SSD",
            "weight": "1.75kg",
            "battery": "8 hours",
            "display": "15.6-inch FHD",
            "os": "Windows 11",
            "price": "₹70,000"
        },

        "Lenovo Legion 5": {
            "processor": "AMD Ryzen 7",
            "ram": "16GB",
            "storage": "1TB SSD",
            "weight": "2.4kg",
            "battery": "6 hours",
            "display": "15.6-inch 165Hz",
            "os": "Windows 11",
            "gpu": "NVIDIA RTX 4060",
            "price": "₹98,000"
        },

        "Lenovo ThinkPad E14": {
            "processor": "Intel i5",
            "ram": "16GB",
            "storage": "512GB SSD",
            "weight": "1.59kg",
            "battery": "10 hours",
            "display": "14-inch FHD",
            "os": "Ubuntu / Windows 11",
            "price": "₹82,000"
        },

        "ASUS TUF Gaming F15": {
            "processor": "Intel i7",
            "ram": "16GB",
            "storage": "1TB SSD",
            "weight": "2.2kg",
            "battery": "7 hours",
            "display": "15.6-inch 144Hz",
            "os": "Windows 11",
            "gpu": "NVIDIA RTX 4050",
            "price": "₹92,000"
        }
    }

    product = mock_db.get(product_name)

    if not product:
        return "Product details not found."

    return (
        f"{product_name} Specifications:\n\n"
        f"• Processor: {product['processor']}\n"
        f"• RAM: {product['ram']}\n"
        f"• Storage: {product['storage']}\n"
        f"• Weight: {product['weight']}\n"
        f"• Battery: {product['battery']}"
    )

print("✅ Product details tool ready")

✅ Product details tool ready


In [4]:
@tool
def compare_products(product1: str, product2: str) -> str:
    """
    Compare two products.

    Args:
        product1: First product
        product2: Second product

    Returns:
        Comparison summary
    """

    return (
        f"Comparison:\n\n"
        f"{product1} vs {product2}\n\n"
        f"• Performance: Comparable\n"
        f"• Battery: {product2} slightly better\n"
        f"• Price: {product1} more affordable\n"
        f"• Recommendation: Choose based on OS preference"
    )

print("✅ Product comparison tool ready")

✅ Product comparison tool ready


In [5]:
import logging
from boto3.session import Session
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType
boto_session = Session()
REGION = boto_session.region_name

In [6]:
logger = logging.getLogger(__name__)

# Memory Creation

In [12]:
# Define memory name for identification
# WHY: so that we can uniquely identify and reuse this memory resource
memory_name = "CustomerSupportMemory" 

# Initialize MemoryManager with region
# WHY: so that we can create/manage memory resources in the correct AWS region
memory_manager = MemoryManager(region_name=REGION)

# Create or fetch existing memory with defined strategies
# WHY: so that we don’t create duplicate memory and can reuse existing one if already present
memory = memory_manager.get_or_create_memory(
    name=memory_name,
    strategies=[
                {
                    # Define USER_PREFERENCE memory strategy
                    # WHY: so that we can store and retrieve user preferences for personalization
                    StrategyType.USER_PREFERENCE.value: {
                        "name": "CustomerPreferences",
                        "description": "Captures customer preferences and behavior",
                        # Define namespace for user-specific preference storage
                        # WHY: so that each user's preferences are isolated using actorId
                        "namespaces": ["support/customer/{actorId}/preferences/"],
                    }
                },
                {
                    # Define SEMANTIC memory strategy
                    # WHY: so that we can store factual information from conversations
                    StrategyType.SEMANTIC.value: {
                        "name": "CustomerSupportSemantic",
                        "description": "Stores facts from conversations",
                        # Define namespace for user-specific semantic memory
                        # WHY: so that conversation facts are stored per user for future retrieval
                        "namespaces": ["support/customer/{actorId}/semantic/"],
                    }
                },
            ]
)

# Extract memory ID from response
# WHY: so that we can reference this memory in agent/runtime later
memory_id = memory["id"]

✅ MemoryManager initialized for region: us-east-1
Created memory: CustomerSupportMemory-l258dZFBS4
Created memory CustomerSupportMemory-l258dZFBS4, waiting for ACTIVE status...
Waiting for memory CustomerSupportMemory-l258dZFBS4 to return to ACTIVE state and strategies to reach terminal states...


[15:56:36]    ⏳ Memory: CREATING, Strategies: 0/2 active (10s elapsed)                             ]8;id=356081;file://c:\Users\lalra\anaconda3\envs\awsbedrockagentcore\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py\manager.py]8;;\:]8;id=387449;file://c:\Users\lalra\anaconda3\envs\awsbedrockagentcore\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py#1023\1023]8;;\

[15:56:47]    ⏳ Memory: CREATING, Strategies: 0/2 active (20s elapsed)                             ]8;id=390068;file://c:\Users\lalra\anaconda3\envs\awsbedrockagentcore\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py\manager.py]8;;\:]8;id=433076;file://c:\Users\lalra\anaconda3\envs\awsbedrockagentcore\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py#1023\1023]8;;\

[15:56:57]    ⏳ Memory: CREATING, Strategies: 0/2 active (31s elapsed)                             ]8;id=408175;file://c:\Users\lalra\anaconda3\envs\awsbedrockagentcore\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py\manager.py]8;;\:]8;id=283921;file://c:\Users\lalra\anaconda3\envs\awsbedrockagentcore\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py#1023\1023]8;;\

[15:57:07]    ⏳ Memory: CREATING, Strategies: 0/2 active (41s elapsed)                             ]8;id=219474;file://c:\Users\lalra\anaconda3\envs\awsbedrockagentcore\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py\manager.py]8;;\:]8;id=68498;file://c:\Users\lalra\anaconda3\envs\awsbedrockagentcore\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py#1023\1023]8;;\

...

Memory CustomerSupportMemory-l258dZFBS4 is ACTIVE and all strategies are in terminal states (took 166 seconds)


              ✅ Memory is ACTIVE (took 166s)                                                       ]8;id=31589;file://c:\Users\lalra\anaconda3\envs\awsbedrockagentcore\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py\manager.py]8;;\:]8;id=495675;file://c:\Users\lalra\anaconda3\envs\awsbedrockagentcore\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py#1043\1043]8;;\

ObservabilityDeliveryManager initialized for region: us-east-1, account: 697629627163
Created log group: /aws/vendedlogs/bedrock-agentcore/memory/APPLICATION_LOGS/CustomerSupportMemory-l258dZFBS4
✅ Logs delivery enabled for memory/CustomerSupportMemory-l258dZFBS4
✅ Traces delivery enabled for memory/CustomerSupportMemory-l258dZFBS4
Observability enabled for memory/CustomerSupportMemory-l258dZFBS4 - logs: True, traces: True


✅ Observability enabled for memory CustomerSupportMemory-l258dZFBS4

Log group: /aws/vendedlogs/bedrock-agentcore/memory/APPLICATION_LOGS/CustomerSupportMemory-l258dZFBS4

In [13]:
if memory_id:
    print("✅ AgentCore Memory created successfully!")
    print(f"Memory ID: {memory_id}")
else:
    print("Memory resource not created. Try Again !")

✅ AgentCore Memory created successfully!
Memory ID: CustomerSupportMemory-l258dZFBS4


## Memory Hook Creation for Persisting Longterm memory

In [14]:
#!/usr/bin/python
"""AgentCore Memory integration for Strands agents."""

import logging
import uuid

import boto3
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType
from boto3.session import Session
# from lab_helpers.utils import get_ssm_parameter, put_ssm_parameter
from strands.hooks import (
    AfterInvocationEvent,
    HookProvider,
    HookRegistry,
    MessageAddedEvent,
)

boto_session = Session()

logger = logging.getLogger(__name__)

ACTOR_ID = "customer_001"
SESSION_ID = str(uuid.uuid4())

memory_client = MemoryClient(region_name=REGION)
memory_name = "CustomerSupportMemory"

class CustomerSupportMemoryHooks(HookProvider):
    """Memory hooks for customer support agent"""

    def __init__(
        self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str
    ):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def retrieve_customer_context(self, event: MessageAddedEvent):
        """Retrieve customer context before processing support query"""
        messages = event.agent.messages
        if (
            messages[-1]["role"] == "user"
            and "toolResult" not in messages[-1]["content"][0]
        ):
            user_query = messages[-1]["content"][0]["text"]

            try:
                all_context = []

                for context_type, namespace in self.namespaces.items():
                    # *** AGENTCORE MEMORY USAGE *** - Retrieve customer context from each namespace
                    memories = self.client.retrieve_memories(
                        memory_id=self.memory_id,
                        namespace=namespace.format(actorId=self.actor_id),
                        query=user_query,
                        top_k=3,
                    )
                    # Post-processing: Format memories into context strings
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(
                                        f"[{context_type.upper()}] {text}"
                                    )

                # Inject customer context into the query
                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0]["text"] = (
                        f"Customer Context:\n{context_text}\n\n{original_text}"
                    )
                    logger.info(f"Retrieved {len(all_context)} customer context items")

            except Exception as e:
                logger.error(f"Failed to retrieve customer context: {e}")

    def save_support_interaction(self, event: AfterInvocationEvent):
        """Save customer support interaction after agent response"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Get last customer query and agent response
                customer_query = None
                agent_response = None

                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif (
                        msg["role"] == "user"
                        and not customer_query
                        and "toolResult" not in msg["content"][0]
                    ):
                        customer_query = msg["content"][0]["text"]
                        break

                if customer_query and agent_response:
                    # *** AGENTCORE MEMORY USAGE *** - Save the support interaction
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[
                            (customer_query, "USER"),
                            (agent_response, "ASSISTANT"),
                        ],
                    )
                    logger.info("Saved support interaction to memory")

        except Exception as e:
            logger.error(f"Failed to save support interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register customer support memory hooks"""
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)
        logger.info("Customer support memory hooks registered")

In [15]:
# Seed with previous customer interactions (Shopping Preference Agent)
previous_interactions = [
    (
        "I'm looking for a lightweight laptop for daily office work. My budget is around ₹80,000 to 1Lakh and I prefer Dell or HP.",
        "USER",
    ),
    (
        "Got it! Based on your preference for lightweight laptops under ₹80,000, Dell Inspiron and HP Pavilion series would be great options. I'll keep your brand and budget preferences in mind for future recommendations.",
        "ASSISTANT",
    ),
    (
        "Can you suggest a good gaming laptop? I don't want to spend more than ₹1 lakh.",
        "USER",
    ),
    (
        "Sure! For gaming under ₹1 lakh, Lenovo Legion and ASUS TUF series are excellent choices. They offer good performance with dedicated GPUs. Let me know if you have any brand preference.",
        "ASSISTANT",
    ),
    (
        "I usually prefer Lenovo laptops. Also, I need at least 16GB RAM.",
        "USER",
    ),
    (
        "Perfect! I'll prioritize Lenovo laptops with 16GB RAM for you. Based on this, Lenovo Legion 5 would be a strong recommendation within your budget.",
        "ASSISTANT",
    ),
    (
        "Show me something similar to what I searched earlier but slightly cheaper.",
        "USER",
    ),
    (
        "Based on your previous preference for Lenovo gaming laptops with 16GB RAM, a slightly more affordable option would be Lenovo IdeaPad Gaming 3. It offers good performance at a lower price point.",
        "ASSISTANT",
    ),
]

In [16]:
# Check if memory_id exists before proceeding
# WHY: so that we don’t attempt to write to memory without a valid memory resource
if memory_id:
    try:
        # Initialize MemoryClient
        # WHAT: low-level client to interact with Bedrock memory service
        # WHY: so that we can write events (conversations) into memory
        memory_client = MemoryClient(region_name=REGION)

        # Create a memory event (store conversation)
        # WHAT: sends past interactions to memory system
        # WHY: so that AWS can process and store them for future retrieval
        memory_client.create_event(
            memory_id=memory_id,  # Reference to the memory resource
            # WHY: tells AWS which memory system to store this data in

            actor_id=ACTOR_ID,  
            # WHAT: unique user identifier
            # WHY: so that memory is stored per user (multi-tenant isolation)

            session_id="previous_session",
            # WHAT: identifier for this conversation session
            # WHY: helps group messages and track conversation context

            messages=previous_interactions,
            # WHAT: list of past user + assistant messages
            # WHY: so that AWS can extract preferences and facts from them
        )

        # Log success message
        # WHY: to confirm that seeding worked successfully
        print("✅ Seeded customer history successfully")

        # Inform that data first goes to Short-Term Memory
        # WHY: events are initially stored as session context before processing
        print("📝 Interactions saved to Short-Term Memory")

        # Inform about async Long-Term Memory processing
        # WHY: AWS will automatically extract preferences + semantic info in background
        print("⏳ Long-Term Memory processing will begin automatically...")

    except Exception as e:
        # Handle any errors during memory write
        # WHY: to avoid crash and help debugging
        print(f"⚠️ Error seeding history: {e}")

✅ Seeded customer history successfully
📝 Interactions saved to Short-Term Memory
⏳ Long-Term Memory processing will begin automatically...


In [17]:
# Import time module
# WHY: so that we can wait between retries while checking async memory processing
import time

# Print initial check message
# WHY: to indicate we are starting to verify Long-Term Memory processing
print("🔍 Checking for processed Long-Term Memories...")

# Initialize retry counter
# WHY: to track how many times we have checked for memory availability
retries = 0

# Set maximum retries (6 × 10 seconds = 60 seconds)
# WHY: to avoid infinite waiting if memory processing is delayed
max_retries = 6  # 1 minute wait

# Start polling loop to check if LTM processing is complete
# WHY: because LTM processing is asynchronous and not immediately available
while retries < max_retries:

    # Retrieve memories from USER_PREFERENCE namespace
    # WHAT: querying extracted preference memories for this user
    # WHY: to verify if AWS has processed STM → LTM
    memories = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace=f"support/customer/{ACTOR_ID}/preferences/",
        query="can you summarize the support issue",
    )

    # Check if any memories are returned
    # WHY: presence of data means LTM processing is complete
    if memories:
        # Print success message with count and time taken
        # WHY: to confirm memory extraction worked
        print(
            f"✅ Found {len(memories)} preference memories after {retries * 10} seconds!"
        )
        break

    # Increment retry count
    # WHY: to move to next polling attempt
    retries += 1

    # If retries still remain
    # WHY: continue waiting for async processing
    if retries < max_retries:
        # Inform user that processing is still ongoing
        print(
            f"⏳ Still processing... waiting 10 more seconds (attempt {retries}/{max_retries})"
        )
        # Wait for 10 seconds before next check
        # WHY: avoid excessive API calls and give time for processing
        time.sleep(10)
    else:
        # Max retries reached
        # WHY: handle delay or failure scenario gracefully
        print(
            "⚠️ Memory processing is taking longer than expected. This can happen with overloading.."
        )
        break

# Print header for extracted preferences
# WHY: to display results in readable format
print(
    "🎯 AgentCore Memory automatically extracted these customer preferences from our seeded conversations:"
)

# Print separator line
# WHY: improve readability of output
print("=" * 80)

# Loop through retrieved memories
# WHY: to display each extracted memory item
for i, memory in enumerate(memories, 1):

    # Ensure memory is in dictionary format
    # WHY: avoid runtime errors if structure differs
    if isinstance(memory, dict):

        # Extract content field
        # WHY: actual memory text is nested inside content
        content = memory.get("content", {})

        # Ensure content is dictionary
        # WHY: safe access to text field
        if isinstance(content, dict):

            # Extract text from memory
            # WHY: this is the actual extracted preference
            text = content.get("text", "")

            # Print memory item
            # WHY: display extracted preference to user
            print(f"  {i}. {text}")

🔍 Checking for processed Long-Term Memories...
⏳ Still processing... waiting 10 more seconds (attempt 1/6)
⏳ Still processing... waiting 10 more seconds (attempt 2/6)
⏳ Still processing... waiting 10 more seconds (attempt 3/6)
⏳ Still processing... waiting 10 more seconds (attempt 4/6)
⏳ Still processing... waiting 10 more seconds (attempt 5/6)
✅ Found 3 preference memories after 50 seconds!
🎯 AgentCore Memory automatically extracted these customer preferences from our seeded conversations:
  1. {"context":"The user explicitly stated they need at least 16GB RAM in a laptop.","preference":"Requires at least 16GB RAM in a laptop","categories":["technology","laptops","hardware"]}
  2. {"context":"The user asked to show something similar but slightly cheaper, indicating a preference for budget-friendly or value-for-money options.","preference":"Prefers value-for-money or budget-friendly laptop options","categories":["technology","laptops","budget"]}
  3. {"context":"The user explicitly men

In [18]:
# Initialize memory client
memory_client = MemoryClient(region_name=REGION)

# Initialize memory hooks
memory_hooks = CustomerSupportMemoryHooks(
    memory_id=memory_id,
    client=memory_client,
    actor_id=ACTOR_ID,
    session_id=SESSION_ID,
)

In [ ]:
import uuid
from strands import Agent
from strands.models import BedrockModel
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager
from strands_agents import (
    SYSTEM_PROMPT,
    search_products,
    get_product_details,
    compare_products,
    MODEL_ID,
)

# Generate a unique session ID
session_id = uuid.uuid4()

# Configure AgentCore memory
# WHAT: defines how agent will interact with memory (read + write)
# WHY: so that agent can retrieve past context and store new information automatically
memory_config = AgentCoreMemoryConfig(
        memory_id=memory_id,
        # WHAT: reference to memory resource created earlier
        # WHY: tells agent which memory system to use

        session_id=str(session_id),
        # WHAT: current session identifier
        # WHY: helps group conversation events in STM

        actor_id=ACTOR_ID,
        # WHAT: user identifier
        # WHY: ensures memory is isolated per user (multi-tenant)

        retrieval_config={
            # Configure semantic memory retrieval
            # WHAT: defines how many semantic memories to fetch and relevance threshold
            # WHY: so that only relevant factual context is passed to LLM
            "support/customer/{actorId}/semantic/": RetrievalConfig(top_k=3, relevance_score=0.2),

            # Configure preference memory retrieval
            # WHAT: defines retrieval behavior for user preferences
            # WHY: so that personalization context is included in LLM prompt
            "support/customer/{actorId}/preferences/": RetrievalConfig(top_k=3, relevance_score=0.2)
        }
    )

# Initialize Bedrock model
# WHAT: creates LLM instance using Claude (or configured model)
# WHY: so that agent can perform reasoning, tool selection, and response generation
model = BedrockModel(model_id="amazon.nova-micro-v1:0", region_name=REGION)

# Create the customer support agent
# WHAT: combines model + memory + tools + system prompt
# WHY: so that we have a fully functional AI agent capable of reasoning, tool usage, and memory handling
agent = Agent(
    model=model,
    session_manager=AgentCoreMemorySessionManager(memory_config, REGION),
    # WHAT: integrates memory into agent runtime
    # WHY: ensures:
    #   - Memory is retrieved before LLM call
    #   - Memory is written after response
    tools=[
        search_products,
        get_product_details,
        compare_products,
    ],
    # hooks=[memory_hooks],
    system_prompt=SYSTEM_PROMPT,
)
print("Agent is all set with LLM, Tool Calls and Memory")

Agent is all set with LLM, Tool Calls and Memory


In [50]:
print("🎧 Testing laptop recommendation with customer memory...\n\n")
response1 = agent("What gaming laptop whould you recommend?")

🎧 Testing laptop recommendation with customer memory...



<thinking>To recommend a gaming laptop, I'll need to first clarify the user's budget and specific requirements. I'll start by asking about the user's budget in Indian Rupees since they mentioned ₹1 lakh, but I'll also confirm if there's a preference for a specific brand or model type like ThinkPad, as they mentioned a preference for ThinkPad models for programming work. Additionally, I'll ask about the minimum requirements for RAM and any other specific needs like Linux compatibility.</thinking>


Hello! To help you find the perfect gaming laptop, could you please clarify the following details for me?

1. What's your preferred budget in Indian Rupees?
2. Do you have a specific brand in mind, such as ThinkPad?
3. What's the minimum amount of RAM you need for programming work?
4. Are there any other specific requirements or features you're looking for, such as Linux compatibility?

In [47]:

response2 = agent("I prefer Macbook with M15 chipset for heavy AIML workload for gaming I like Asus Gaming Laptops")



<thinking>I need to find suitable gaming laptops with AI/ML capabilities that fit the budget. I will use the search_products tool to find the products and then provide details if needed. Since the user has mentioned preferences, I will make sure to consider those.</thinking>


Tool #1: search_products
category = laptops
budget = up to 100000
brand = Asus
<thinking>It appears there were no Asus gaming laptops available within the specified budget range that are suitable for AI/ML workloads. I should look for gaming laptops from other brands that fit the user's preferences and budget.</thinking>

Tool #2: search_products

Tool #3: search_products

Tool #4: search_products
category = laptops
budget = up to 100000
brand = Dell
category = laptops
budget = up to 100000
brand = HP
category = laptops
budget = up to 100000
brand = Lenovo
<thinking>I found three gaming laptops from Dell, HP, and Lenovo that fit within the budget. I'll provide details about each and offer to compare them if the

In [ ]:
print("Compare product with customer memory...\n\n")
response2 = agent("What brand I prefer?")

In [ ]:
print("Compare product with customer memory...\n\n")
response3 = agent("Compare the laptop 1 and 2")